In [46]:
import os
import pickle
import jsonlines
import pandas as pd
import numpy as np
import json
import copy
from tqdm import tqdm

In [47]:
# meta_files = ['./raw/yelp_academic_dataset_business.json', './raw/yelp_academic_dataset_checkin.json', './raw/yelp_academic_dataset_review.json', './raw/yelp_academic_dataset_tip.json', './raw/yelp_academic_dataset_user.json']
# for meta_file in meta_files:
#     lines = open(meta_file).readlines()
#     for line in tqdm(lines):
#         info = json.loads(line)
#         if 'rncjoVoEFUJGCUoC1JgnUA' == info['business_id']:
#             print(f"found {info['business_id']} in {meta_file}")

In [48]:
data = json.load(open("./handled/item2attributes.json", "r"))

In [49]:
example_dict = {}
for item_dict in tqdm(data.values()):
    example_dict.update(item_dict)

print(example_dict)

100%|██████████| 36673/36673 [00:00<00:00, 882381.99it/s]

{'business_id': 'WnT9NIzQgLlILjPT0kEcsQ', 'name': 'Adelita Taqueria & Restaurant', 'address': '1108 S 9th St', 'city': 'Philadelphia', 'state': 'PA', 'postal_code': '19147', 'latitude': 39.9359818, 'longitude': -75.158665, 'stars': 4.5, 'review_count': 35, 'is_open': 1, 'attributes': {'WheelchairAccessible': 'False', 'RestaurantsReservations': 'False', 'Caters': 'True', 'BusinessAcceptsCreditCards': 'True', 'HasTV': 'True', 'GoodForKids': 'True', 'BikeParking': 'True', 'WiFi': "u'free'", 'Alcohol': "u'none'", 'RestaurantsAttire': "'casual'", 'RestaurantsTakeOut': 'True', 'RestaurantsTableService': 'True', 'Ambience': "{'touristy': None, 'hipster': False, 'romantic': None, 'divey': None, 'intimate': None, 'trendy': None, 'upscale': None, 'classy': None, 'casual': True}", 'RestaurantsGoodForGroups': 'True', 'BusinessParking': "{'garage': None, 'street': True, 'validated': None, 'lot': False, 'valet': False}", 'GoodForMeal': "{'dessert': True, 'latenight': None, 'lunch': True, 'dinner': T

In [50]:
cate_dict = {}
for item_dict in tqdm(data.values()):
    if not "categories" in item_dict.keys():
        print(f"No categories for item: {item_dict}")
    # if item_dict["categories"] and len(item_dict["categories"]) >= 1:
    cate_dict[item_dict["business_id"]] = item_dict["categories"]
    # else:
    #     cate_dict[item_dict["business_id"]] = "NA"


100%|██████████| 36673/36673 [00:00<00:00, 775228.36it/s]


去掉longitude和latitude两个属性，剩下的属性可以分为文本类和列表类

文本类：直接添加到prompt即可

列表类：先把列表中的element组成文本，再添加到prompt

In [51]:
# instruction = "The point of interest has the following attributes: \n "
instruction = "The point of interest has the following attributes: \n "
image_instruction = "The point of interest has the image: "

In [52]:
# {"photo_id": "zsvj7vloL4L5jhYyPIuVwg", "business_id": "Nk-SJhPlDBkAZvfsADtccA", "caption": "Nice rock artwork everywhere and craploads of taps.", "label": "inside"}
# {"photo_id": "HCUdRJHHm_e0OCTlZetGLg", "business_id": "yVZtL5MmrpiivyCIrVkGgA", "caption": "", "label": "outside"}
# {"photo_id": "vkr8T0scuJmGVvN2HJelEA", "business_id": "_ab50qdWOk0DdB6XOrBitw", "caption": "oyster shooter", "label": "drink"}
# {"photo_id": "pve7D6NUrafHW3EAORubyw", "business_id": "SZU9c8V2GuREDN5KgyHFJw", "caption": "Shrimp scampi", "label": "food"}
import json
from collections import defaultdict


# photo_id_map_path = "./raw/photos.json"

# business_to_inside_photos = defaultdict(list)

# with open(photo_id_map_path, 'r') as f:
#     for line in f:
#         photo = json.loads(line)
#         if photo.get("label") == "food":
#             business_id = photo.get("business_id")
#             photo_id = photo.get("photo_id")
#             if business_id and photo_id:
#                 business_to_inside_photos[business_id].append(photo_id)
# print(f"length of business_to_inside_photos: {len(business_to_inside_photos)}")
# # 示例输出
# for business_id, photo_ids in list(business_to_inside_photos.items())[:3]:
#     print(f"{business_id}: {photo_ids}")



photo_id_map_path = "./raw/photos.json"

# 存储每个 business 的不同类型图片
business_photos = defaultdict(lambda: defaultdict(list))

with open(photo_id_map_path, 'r') as f:
    for line in f:
        photo = json.loads(line)
        business_id = photo.get("business_id")
        photo_id = photo.get("photo_id")
        label = photo.get("label")

        if business_id and photo_id:
            business_photos[business_id][label].append(photo_id)
print(f"Number of businesses with photos: {len(business_photos)}")
# 选出优先图：inside 优先，其次 food
business_to_preferred_photo = {}
for business_id, label_dict in business_photos.items():
    if label_dict["food"]:
        business_to_preferred_photo[business_id] = label_dict["food"][0]
    elif label_dict["inside"]:
        business_to_preferred_photo[business_id] = label_dict["inside"][0]
    elif label_dict["drink"]:
        business_to_preferred_photo[business_id] = label_dict["drink"][0] 
    else:
        # print(f"No preferred photo for business: {business_id}, labels: {label_dict.keys()[0]}")
        business_to_preferred_photo[business_id] = label_dict[list(label_dict.keys())[0]][0] if label_dict else None

print(f"Number of businesses with preferred photo: {len(business_to_preferred_photo)}")

# 示例输出前 3 条
for business_id, photo_id in list(business_to_preferred_photo.items())[:3]:
    print(f"{business_id}: {photo_id}")
save_path = "./handled/business_to_preferred_photo.json"
with open(save_path, 'w') as f:
    json.dump(business_to_preferred_photo, f, indent=4)

with open("./handled/business_to_preferred_photo.json", "r") as f:
    business_to_preferred_photo = json.load(f)

Number of businesses with photos: 36680
Number of businesses with preferred photo: 36680
Nk-SJhPlDBkAZvfsADtccA: ghASKsz6wb39srM7s6nYJA
yVZtL5MmrpiivyCIrVkGgA: HCUdRJHHm_e0OCTlZetGLg
_ab50qdWOk0DdB6XOrBitw: MG-9Qu4DLqmpXo-HSiGRew


In [53]:
# import copy
import ast



def dealWithDict(item_dict):
    item_dict = copy.deepcopy(item_dict)
    item_str = ""
    for key, value in item_dict.items():


        if key in ["Ambience", "Music", "BusinessParking", "GoodForMeal", "BestNights"]:
            
            value = ast.literal_eval(value)
            if value is None:
                continue
            found = False
            for k, v in value.items():
                if v == True:
                    value = k
                    found = True
                    break
            if not found:
                continue
        elif isinstance(value, dict):
            value = dealWithDict(value)
        
        item_str += f"{key} is {value}, "
    return item_str



# print(dict2str(dict))

In [54]:
item_data = {}
limited = 1000
count = 0
for item_dict in tqdm(data.values()):
    # count += 1
    # if count > limited:
    #     break
    item_prompt = copy.deepcopy(instruction)
    item_id = None
    for key, value in item_dict.items():
        
        if value is None:
            continue
        if key in ["longitude", "latitude", "hours"]:   # drop longitude and latitude
            continue
        elif key in ["business_id"]:  # get the item id
            item_id = value
        elif key in ["categories", "neighborhoods"]:    # list type attributes
            attri_str = ""
            
            
            if len(value) == 0:
                attri_str = "none, "
            else:
                # for meta_str in value:
                #     attri_str += (meta_str + ", ")
                # print(value)
                attri_str += value
            attri_prompt = key + " is " + attri_str[:-2] + "; "    # [:-2] is to remove the last ", "
            item_prompt += attri_prompt
        elif key in ["Ambience", "Music", "BusinessParking"]:
            for k, v in value.items():
                if v == True:
                    attri_prompt = key + " is " + k + "; "
                    break
            item_prompt += attri_prompt
        elif isinstance(value, dict): #key in ["attributes", "BusinessParking", "GoodForMeal", "BestNights"]:  # dict type attributes
            value = dealWithDict(value)[:-2]  # remove the last ", "
            attri_prompt = key + " is " + value + "; "
            item_prompt += attri_prompt
        else:   # str type attributes 
            attri_prompt = key + " is " + str(value).replace("\n", ", ") + "; "
            item_prompt += attri_prompt
    if item_id:
        item_data[item_id] = item_prompt[:-2]
    else:
        raise ValueError("No item id")

100%|██████████| 36673/36673 [00:03<00:00, 12095.12it/s]


In [55]:
count = 0
for item_id, item_prompt in tqdm(item_data.items()):
    if item_id not in business_to_preferred_photo:
        print(f"Item {item_id} does not have a preferred photo.")
        count += 1

print(f"Number of items without preferred photo: {count}")

100%|██████████| 36673/36673 [00:00<00:00, 787141.65it/s]

Number of items without preferred photo: 0


In [56]:
# json.dump(item_data, open("./handled/item_str.json", "w"))

In [57]:
# convert to jsonline
def save_data(data_path, data):
    '''write all_data list to a new jsonl'''
    with jsonlines.open("./handled/"+ data_path, "w") as w:
        for meta_data in data:
            w.write(meta_data)

id_map = json.load(open("./handled/id_map.json", "r"))["item2id"]
json_data = []
for key, value in item_data.items():
    if key not in id_map:
        print(f"Item {key} not found in id_map, skipping.")
        continue
    image_name = business_to_preferred_photo.get(key, "")
    if image_name:
        image_path = f"{image_name}.jpg"
    else:
        image_path = ""
    json_data.append({"attributes_input": value, "image_instruction_input": image_instruction, "target": "", "image_url": image_path, "item": key, "item_id": int(id_map[key])})
json_data.sort(key=lambda x: x["item_id"])
save_data("multimodal_item_str.jsonline", json_data)

Item CF33F8-E6oudUQ46HnavjQ not found in id_map, skipping.
Item bBDDEgkFA1Otx9Lfe7BZUQ not found in id_map, skipping.
Item eEOYSgkmpB90uNA7lDOMRA not found in id_map, skipping.
Item il_Ro8jwPlHresjw9EGmBg not found in id_map, skipping.
Item ABxoFuzZy5mqQ8C5FJJajQ not found in id_map, skipping.
Item 0qNpTGTcqPwOLi2hADx4Xw not found in id_map, skipping.
Item aCDY7vXYMs54EbYuQScsnQ not found in id_map, skipping.
Item NZ_bFJma7brQUfln5h1UAg not found in id_map, skipping.
Item RK6-cJ9hj53RzOlCBmpT-g not found in id_map, skipping.
Item Fk1xpM7fjqEJ4paqk5ZFSg not found in id_map, skipping.
Item zFvqulgAYOpSG2t1v8AZ-w not found in id_map, skipping.
Item QjV4v7q_pt7tt3K1US7IHg not found in id_map, skipping.
Item tSFXJ0GFl5iUdy021YgWLw not found in id_map, skipping.
Item Y6heWJJ9AmEL58fZwgi9YQ not found in id_map, skipping.
Item jL_NufxqXi-BpW5uXKsPwQ not found in id_map, skipping.
Item xa6JYHDgVza7CuenKBJBHw not found in id_map, skipping.
Item rZw9O5lJ36m_mXeRKE4G9A not found in id_map, skippin

In [58]:
# {"attributes_input": "The business has the following attributes: \n name is St Honore Pastries; address is 935 Race St; city is Philadelphia; state is PA; postal_code is 19107; stars is 4.0; review_count is 80; is_open is 1; attributes is {'RestaurantsDelivery': 'False', 'OutdoorSeating': 'False', 'BusinessAcceptsCreditCards': 'False', 'BusinessParking': \"{'garage': False, 'street': True, 'validated': False, 'lot': False, 'valet': False}\", 'BikeParking': 'True', 'RestaurantsPriceRange2': '1', 'RestaurantsTakeOut': 'True', 'ByAppointmentOnly': 'False', 'WiFi': \"u'free'\", 'Alcohol': \"u'none'\", 'Caters': 'True'}; categories is Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeri; hours is {'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', 'Wednesday': '7:0-20:0', 'Thursday': '7:0-20:0', 'Friday': '7:0-21:0', 'Saturday': '7:0-21:0', 'Sunday': '7:0-21:0'}", "image_instruction_input": "The business has the image: ", "target": "", "image_url": "N3tTxsgpFH91wzo4gZm19A.jpg", "item": "MTSW4McQd7CbVtyjqoe9mw", "item_id": 20459}

# {"attributes_input": "The point of interest has the following attributes: \n name is St Honore Pastries; address is 935 Race St; city is Philadelphia; state is PA; postal_code is 19107; stars is 4.0; review_count is 80; is_open is 1; attributes is RestaurantsDelivery is False, OutdoorSeating is False, BusinessAcceptsCreditCards is False, BusinessParking is BusinessParking is street, , BikeParking is True, RestaurantsPriceRange2 is 1, RestaurantsTakeOut is True, ByAppointmentOnly is False, WiFi is u'free', Alcohol is u'none', Caters is True; categories is Restaurants, Food, Bubble Tea, Coffee & Tea, Bakeri", "image_instruction_input": "The point of interest has the image: ", "target": "", "image_url": "N3tTxsgpFH91wzo4gZm19A.jpg", "item": "MTSW4McQd7CbVtyjqoe9mw", "item_id": 20459}
